In [ ]:
# Day 2 - Self-Attention from Scratch + Small Model Training
# LLM from Scratch Journey

import torch
import torch.nn as nn
import torch.nn.functional as F

print('Torch version:', torch.__version__)
print('GPU Available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# ====================== 1. Simple Self-Attention ======================
class SimpleSelfAttention(nn.Module):
    def __init__(self, head_size, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)      # (B, T, head_size)
        q = self.query(x)    # (B, T, head_size)
        v = self.value(x)
        
        # Scaled dot-product attention
        scores = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)   # (B, T, T)
        attn = F.softmax(scores, dim=-1)
        out = attn @ v                                             # (B, T, head_size)
        return out

# Test
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
attn = SimpleSelfAttention(head_size=16, n_embd=C)
out = attn(x)
print('Input shape:', x.shape)
print('Output shape:', out.shape)

In [ ]:
# ====================== 2. Load Shakespeare Dataset ======================
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

print('Length of dataset:', len(text))
print('First 300 characters:\n', text[:300])

In [ ]:
# Create vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('Vocabulary size:', vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s): 
    return [stoi[c] for c in s]

def decode(l): 
    return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print('Data tensor shape:', data.shape)

In [ ]:
# ====================== 3. Data Loader ======================
block_size = 32   # context length
batch_size = 64

def get_batch(split):
    n = int(0.9 * len(data))
    train_data = data[:n]
    val_data = data[n:]
    
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('Batch shapes:', xb.shape, yb.shape)

In [ ]:
# ====================== 4. Bigram Model ======================
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)   # (B, T, vocab_size)
        
        if targets is None:
            return logits
        
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        return logits, loss

model = BigramLanguageModel(vocab_size).to(device)
print('Bigram Model created!')

In [ ]:
# ====================== 5. Training ======================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
epochs = 3000

for epoch in range(epochs):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f'Epoch {epoch:4d} | Loss: {loss.item():.4f}')

In [ ]:
# ====================== 6. Generate Text ======================
@torch.no_grad()
def generate_text(model, start_text="Once upon a time", max_new_tokens=300):
    model.eval()
    context = torch.tensor(encode(start_text), dtype=torch.long, device=device).unsqueeze(0)
    generated = model.generate(context, max_new_tokens) if hasattr(model, 'generate') else None
    # For now we'll implement simple generation
    for _ in range(max_new_tokens):
        logits = model(context[:, -block_size:])
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        context = torch.cat((context, next_token), dim=1)
    return decode(context[0].tolist())

print(generate_text(model))